In [5]:
import openai
from tqdm.auto import tqdm
import time
import os
openai.api_key = os.environ["OPENAI_API_KEY"]

/mount/arbeitsdaten/asr-2/vaethdk/virtualenvs/cts_en/lib64/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import sys
sys.path.append('../../..')
print(os.path.realpath("."))

from data.dataset import ReimburseGraphDataset, StandardGraphDataset, DataAugmentationLevel, NodeType, DialogNode, Question

/mount/arbeitsdaten41/projekte/asr-2/vaethdk/cts_newcodebase_rollback/conversational-tree-search/generation/reimburse/chatgpt


In [9]:
onboard_human_data = StandardGraphDataset('en/onboarding/train_graph.json', 'en/onboarding/train_answers.json', True, DataAugmentationLevel.NONE, augmentation_path=None, resource_dir='../../../resources')
reimburse_human_data = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/train_answers.json', True, DataAugmentationLevel.NONE, augmentation_path=None, resource_dir='../../../resources')

===== Dataset Statistics =====
- files:  en/onboarding/train_graph.json en/onboarding/train_answers.json
- synonyms: True
- depth: 12  - degree: 9
- answers: 43
- questions: 141
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  4
- answer limit: 0  - maximum loaded:  1
===== Dataset Statistics =====
- files:  en/reimburse/train_graph.json en/reimburse/train_answers.json
- synonyms: True
- depth: 20  - degree: 13
- answers: 248
- questions: 279
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  7
- answer limit: 0  - maximum loaded:  9


In [45]:
def prompt(node_text: str, num_questions: int):
    return f"""Generate {num_questions} questions about the given facts: "{node_text}"""

def api_prompt(prompt: str):
    return [
        {"role": "system", "content": "You are a truthful assistant, generating diverse FAQ-style questions given some facts. The generated questions should be answerable using the given fact only, without additional knowledge. The questions should also be short and human-like. Try to vary the amount of information between questions. Present the results in a numbered list."},
        {"role": "user", "content": prompt},
    ]

def api_completion(node_text: str, num_questions: int):
    return openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=api_prompt(prompt(node_text, num_questions))
    )

In [46]:
# NUM_QUESTIONS = 10 

# for node in tqdm(human_data.nodes_by_type[NodeType.INFO]):
#     result = api_completion(node.text, NUM_QUESTIONS)
#     break


In [47]:
# print(result)

In [10]:
def parse_output(result):
    result_strings = result.get('choices')[0].get("message").get("content").split('\n')
    questions = []
    unnumbered_questions = []
    question_idx = 1
    for question in result_strings:
        question = question.strip()
        if question.startswith(f"{question_idx}."):
            questions.append(question.strip(f"{question_idx}.").strip())
        else:
            unnumbered_questions.append(question)
        question_idx += 1
    return questions, unnumbered_questions

In [49]:
# good, bad = parse_output(result)
# print(good)

In [52]:
import traceback

NUM_QUESTIONS = 10 

generated = {}
generated_unnumbered = {}

num_generated = 0
num_generated_unnumbered = 0

for idx, node in tqdm(enumerate(human_data.nodes_by_type[NodeType.INFO])):
    done = False
    while not done:
        try:
            response = api_completion(node.text, NUM_QUESTIONS)
            questions, unnumbered_questions = parse_output(response)

            generated[node.key] = questions
            generated_unnumbered[node.key] = unnumbered_questions

            num_generated += len(questions)
            num_generated_unnumbered += len(unnumbered_questions)

            if idx % 10 == 0:
                print(f"Generated: {num_generated}, Unnumbered: {num_generated_unnumbered}")
            
            done = True
        except:
            traceback.print_exc()
            done = True
            print("waiting...")
            time.sleep(15)
    break

0it [00:30, ?it/s]

Generated: 10, Unnumbered: 0


In [53]:
# import json
# with open("../../../resources/en/reimburse/generated/chatgpt/train_questions_v2.json", "w") as f:
#     json.dump(generated, f)

# with open("../../../resources/en/reimburse/generated/chatgpt/train_questions_v2_unnumbered.json", "w") as f:
#     json.dump(generated_unnumbered, f)

# Generate Answers

In [11]:
onboard_human_data = StandardGraphDataset('en/onboarding/train_graph.json', 'en/onboarding/train_answers.json', True, DataAugmentationLevel.NONE, augmentation_path=None, resource_dir='../../../resources')
reimburse_human_data = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/train_answers.json', True, DataAugmentationLevel.NONE, augmentation_path=None, resource_dir='../../../resources')

===== Dataset Statistics =====
- files:  en/onboarding/train_graph.json en/onboarding/train_answers.json
- synonyms: True
- depth: 12  - degree: 9
- answers: 43
- questions: 141
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  4
- answer limit: 0  - maximum loaded:  1
===== Dataset Statistics =====
- files:  en/reimburse/train_graph.json en/reimburse/train_answers.json
- synonyms: True
- depth: 20  - degree: 13
- answers: 248
- questions: 279
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  7
- answer limit: 0  - maximum loaded:  9


In [38]:
def prompt(node_text: str, answer_text: str, num_paraphrases: int):
    return f"""Generate {num_paraphrases} answer paraphrases for the answer "{answer_text}" to the question: "{node_text}"""

def api_prompt(prompt: str):
    return [
        {"role": "system", "content": "You are a truthful assistant, generating diverse paraphrases for a prototypical answer to a given question. The generated answer paraphrases should be human-like and preferably short. Present the results in a numbered list."},
        {"role": "user", "content": prompt},
    ]

def api_completion(node_text: str, answer_text: str, num_paraphrases: int):
    return openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=api_prompt(prompt(node_text, answer_text, num_paraphrases))
    )

In [39]:
from collections import defaultdict
import traceback

NUM_PARAPHRASES = 10 

generated = defaultdict(lambda: set())
generated_unnumbered = defaultdict(lambda: set())

num_generated = 0
num_generated_unnumbered = 0

for idx, node in tqdm(enumerate(onboard_human_data.nodes_by_type[NodeType.QUESTION])):
    for answer in node.answers:
        done = False
        while not done:
            try:
                print(api_prompt(prompt(node.text, answer.text, NUM_PARAPHRASES)))
                response = api_completion(node.text, answer.text, NUM_PARAPHRASES)
                answers, unnumbered_answers = parse_output(response)

                generated[answer.text.strip().lower()] = generated[answer.text.strip().lower()].union(answers)
                generated_unnumbered[answer.text.strip().lower()] = generated_unnumbered[answer.text.strip().lower()].union(unnumbered_answers)

                num_generated += len(answers)
                num_generated_unnumbered += len(unnumbered_answers)

                if idx % 10 == 0:
                    print(f"Generated: {num_generated}, Unnumbered: {num_generated_unnumbered}")
                
                done = True
            except:
                traceback.print_exc()
                done = True
                print("waiting...")
                time.sleep(15)

0it [00:00, ?it/s]

[{'role': 'system', 'content': 'You are a truthful assistant, generating diverse paraphrases for a prototypical answer to a given question. The generated answer paraphrases should be human-like and preferably short. Present the results in a numbered list.'}, {'role': 'user', 'content': 'Generate 10 answer paraphrases for the answer "Finding Housing" to the question: "Hi I\'m the Stuttgart Help-Bot! My goal is to help your move to Germany and especially to Stuttgart go as smoothly as possible!\n What would you like to know about?'}]


0it [00:13, ?it/s]

Generated: 10, Unnumbered: 0


In [40]:
generated

defaultdict(<function __main__.<lambda>()>,
            {'finding housing': {'"Discovering Housing Opportunities"',
              '"Exploring Housing Options"',
              '"Finding a Place to Live"',
              '"Hunting for a Place in Stuttgart"',
              '"Inquiries About Housing in Stuttgart"',
              '"Looking for a Place to Stay"',
              '"Scouting for a Residence"',
              '"Searching for Accommodation"',
              '"Seeking Housing Solutions"',
              '"Seeking a Home in Stuttgart"'}})

In [28]:
generated_unnumbered

defaultdict(<function __main__.<lambda>()>, {})

In [24]:
api_prompt(prompt(node_text, answer_text, num_paraphrases))

NameError: name 'node_text' is not defined